<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/_banners/opim5641_banner.svg" width="100%" alt="OPIM 5641 banner"/>

**Dr. Dave Wanik - Operations and Information Management - University of Connecticut**

---

# Assignment 6 — Integer Programming: Pick a Fantasy Football Team

In M5.1 you used binary variables to choose projects: $y = 1$ if we do the project, $y = 0$ if we do not. Now
the projects are football players. In a fantasy football contest you pick a team of real NFL players, and your
team scores points based on how those players do in that week's real games. Every player has a salary, and
your whole team has to fit under a salary cap of &#36;50,000.

Our own Prof. David Bergman [won &#36;2.5 million playing this game](https://today.uconn.edu/2021/01/business-professor-wins-2-5-million-fantasy-football-jackpot-using-concepts-teaches-students/)
with the same ideas he teaches his students.

**You do not write any model code.** The model is already here, wrapped up so that a whole scenario is
**one line**. You work through the steps top to bottom: run a cell, read what it prints, and fill in the blanks.

| Step | What you do | Points |
|---|---|---|
| 1 | Load the players (just run it) | - |
| 2 | Load the model (just run it) | - |
| 3 | The base model: the best nine players | 20 |
| 4 | Change the salary cap | 20 |
| 5 | Add business rules | 30 |
| 6 | Pick the team before the games are played | 15 |
| 7 | Your recommendation | 15 |

### The model you are solving

Each team has exactly nine players: 2 fullbacks (FB), 2 running backs (RB), 3 wide receivers (WR) and 2 tight
ends (TE).

**Decision variables**

$y_i = 1$ if we pick player $i$, and $y_i = 0$ if we do not.

**Objective function**

$$\max \sum_{i} \text{points}_i \, y_i$$

**Constraints**

$$
\begin{aligned}
\sum_{i \in FB} y_i = 2, \quad \sum_{i \in RB} y_i = 2, \quad \sum_{i \in WR} y_i = 3, \quad \sum_{i \in TE} y_i = 2 && \text{(the positions)} \\
\sum_{i} \text{salary}_i \, y_i \le 50{,}000 && \text{(the salary cap)} \\
y_i \in \{0, 1\} \quad \text{for every player } i && \text{(a player is on the team or not)}
\end{aligned}
$$

## Import modules (download Pyomo and the cbc solver)

Run this first, every time. It installs Pyomo and **cbc**, the solver for models with binary variables. It takes
about a minute.

In [ ]:
# install Pyomo and the cbc solver in Colab (idaes-pse brings both)
import sys
import os

if 'google.colab' in sys.modules:
    !pip install -q idaes-pse --pre
    !idaes get-extensions --to ./bin
    os.environ['PATH'] += ':bin'

%matplotlib inline
from pylab import *
import numpy as np
import pandas as pd
from pyomo.environ import *

# where the setup cell put the solver
cbc_path = '/content/bin/cbc'

# Step 1 · Load the players

**Just run it.** The file has one row per player per week for Weeks 2 through 10 of the 2021 NFL season: name,
position, team, opponent, fantasy salary and fantasy points. We are playing the **Week 10** contest.

Three players have a salary of &#36;0 in Week 10, which means they were not in the contest, so the cell shows you
those rows and drops them.

In [ ]:
all_weeks = pd.read_csv('https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/Assignments/data/nfl_fantasy_2021.csv')
print(len(all_weeks), "rows in the file")

# we are playing the Week 10 contest
players = all_weeks[all_weeks['Week'] == 10]
print(len(players), "players in Week 10")

# the rows with no salary
no_salary = players[players['Salary'] == 0]
print("no salary:")
print(no_salary)

players = players[players['Salary'] > 0]
players = players.reset_index(drop=True)
print()
print(len(players), "players after dropping them")

players.head()

In [ ]:
# how many players are at each position?
players['Position'].value_counts()

In [ ]:
# look before you model: do the expensive players score more?
scatter(players['Salary'], players['Points'], alpha=0.4, color='black')
xlabel('Salary ($)')
ylabel('Fantasy points in Week 10')
title('382 players')
show()

# Step 2 · Load the model

**Just run this cell.** The code is hidden because you do not need to change it (double-click the title to see
it). It is the model at the top of this notebook: one binary variable per player, one constraint per position
and one salary cap, plus the business rules you will switch on in Step 5.

After this cell, a whole scenario is one line:

`pick_team("a name", salary_cap=50000)`

It prints the team, its points and its salary, and it adds the scenario to a scoreboard. `show_all()` prints the
scoreboard.

In [ ]:
#@title Step 2 · Load the model (just run this cell - the code is hidden on purpose) { display-mode: "form" }
import logging
from math import comb
from IPython.display import display
logging.getLogger('pyomo.core').setLevel(logging.ERROR)

roster_rules = {"FB": 2, "RB": 2, "WR": 3, "TE": 2}
scoreboard = {}
indices = players.index.values.tolist()
Points = players['Points'].tolist()
Salary = players['Salary'].tolist()
Position = players['Position'].tolist()
Team = players['Team'].tolist()
Name = players['Player'].tolist()


def pick_team(scenario, salary_cap=50000, whole_players=True, max_per_team=None, must_start=None,
              not_together=None, requires=None, forecast=False):
    # empty rule lists when a rule is not switched on
    if must_start is None:
        must_start = []
    if not_together is None:
        not_together = []
    if requires is None:
        requires = []

    # check the names first, so a typo gets a helpful message
    names_used = list(must_start)
    for pair in not_together + requires:
        names_used.append(pair[0])
        names_used.append(pair[1])
    for name in names_used:
        if name not in Name:
            print("I cannot find", repr(name), "- check the spelling against show_players()")
            return

    model = ConcreteModel()

    # decision variables: one per player
    if whole_players:
        model.y = Var(indices, domain=Binary)
    else:
        model.y = Var(indices, domain=UnitInterval)      # fractions of a player are allowed

    model.constraints = ConstraintList()

    # the positions
    for position in roster_rules:
        expr = 0
        for i in indices:
            if Position[i] == position:
                expr += model.y[i]
        model.constraints.add(expr == roster_rules[position])

    # the salary cap
    expr = 0
    for i in indices:
        expr += Salary[i] * model.y[i]
    model.constraints.add(expr <= salary_cap)

    # at most n: no more than max_per_team players from one NFL team
    if max_per_team is not None:
        for team in set(Team):
            expr = 0
            for i in indices:
                if Team[i] == team:
                    expr += model.y[i]
            model.constraints.add(expr <= max_per_team)

    # must start: y = 1
    for name in must_start:
        model.constraints.add(model.y[Name.index(name)] == 1)

    # mutually exclusive: y_a + y_b <= 1
    for pair in not_together:
        model.constraints.add(model.y[Name.index(pair[0])] + model.y[Name.index(pair[1])] <= 1)

    # contingency: the first player requires the second, so y_second - y_first >= 0
    for pair in requires:
        model.constraints.add(model.y[Name.index(pair[1])] - model.y[Name.index(pair[0])] >= 0)

    # the objective: the points we believe in
    if forecast:
        believed = players['Forecast'].tolist()
    else:
        believed = Points
    expr = 0
    for i in indices:
        expr += believed[i] * model.y[i]
    model.total = Objective(expr=expr, sense=maximize)

    results = SolverFactory('cbc', executable=cbc_path).solve(model)
    status = str(results.solver.termination_condition)
    print("SCENARIO:", scenario, "   (solver:", status, ")")
    if status != "optimal":
        print("no team can meet all of these rules")
        return

    # the team
    picked = []
    share = []
    salary_used = 0
    points_scored = 0
    for i in indices:
        if model.y[i]() > 0.001:
            picked.append(i)
            share.append(round(model.y[i](), 3))
            salary_used += Salary[i] * model.y[i]()
            points_scored += Points[i] * model.y[i]()

    columns = ['Position', 'Player', 'Team', 'Salary', 'Points']
    if forecast:
        columns.append('Forecast')
    roster = players.loc[picked, columns].copy()
    if not whole_players:
        roster['Share of player'] = share
    display(roster.sort_values('Position'))

    if forecast:
        print("the model expected {:.1f} points".format(model.total()))
    print("points actually scored = {:.1f}".format(points_scored))
    unspent = max(salary_cap - salary_used, 0)
    print("salary used = ${:,.0f} of ${:,.0f} (${:,.0f} unspent)".format(salary_used, salary_cap, unspent))

    scoreboard[scenario] = {'points scored': round(points_scored, 1), 'salary used': round(salary_used),
                            'salary cap': salary_cap}


def show_all():
    display(pd.DataFrame(scoreboard).T)
    names = list(scoreboard.keys())
    scores = []
    for name in names:
        scores.append(scoreboard[name]['points scored'])
    figure(figsize=(10, 1.5 + 0.45 * len(names)))
    barh(names, scores, color='#E0701A')
    gca().invert_yaxis()
    xlabel('Fantasy points actually scored')
    title('Same players, different choices')
    show()


def count_teams():
    total = 1
    for position in roster_rules:
        n_at_position = Position.count(position)
        total = total * comb(n_at_position, roster_rules[position])
    print("binary decision variables:", len(indices))
    print("different teams with the right positions: {:,}".format(total))
    print("years to check them all at a million teams a second: {:.0f}".format(total / 1000000 / 3600 / 24 / 365.25))


def show_players(position):
    table = players[players['Position'] == position]
    table = table.sort_values('Salary', ascending=False)
    display(table[['Position', 'Player', 'Team', 'Salary', 'Points']].head(15))


print("the model is loaded")

# Step 3 · The base model: the best nine players (20 points)

Nine players, the four position rules and the &#36;50,000 cap. Run the three cells.

In [ ]:
pick_team("1 base model", salary_cap=50000)

In [ ]:
count_teams()

In [ ]:
# every fullback in the contest
players[players['Position'] == 'FB']

Now remove the integer rule. `whole_players=False` lets $y_i$ be any number from 0 to 1, so the model can buy
part of a player.

In [ ]:
pick_team("2 fractions allowed", salary_cap=50000, whole_players=False)

### YOUR ANSWER - Step 3

Double-click this cell and fill in the blanks.

**3a (5 points).** The base team scores \_\_\_\_\_\_ points and costs &#36;\_\_\_\_\_\_, which leaves &#36;\_\_\_\_\_\_ unspent.

**3b (5 points).** The model has \_\_\_\_\_\_ binary decision variables, and there are about \_\_\_\_\_\_ different teams
with the right positions. Why is the brute force method from Module 2 not an option here? One sentence.

*your sentence*

**3c (5 points).** Look at the two fullbacks on the base team. They scored very little. Why did the model pick
them, and why these two? (The table above shows every fullback.) One or two sentences.

*your sentences*

**3d (5 points).** With fractions allowed, the team scores \_\_\_\_\_\_ points. Which two players were split, and
what share of each did the model buy? If you round each share to the nearest whole player, is the team still
under the cap? Why does a whole-number rule never improve the objective? Two or three sentences.

*your sentences*

# Step 4 · Change the salary cap (20 points)

What would more money buy? Run all four caps, then `show_all()` puts every scenario so far in one table and one
picture.

In [ ]:
pick_team("3 cap 40K", salary_cap=40000)

In [ ]:
pick_team("4 cap 45K", salary_cap=45000)

In [ ]:
pick_team("5 cap 55K", salary_cap=55000)

In [ ]:
pick_team("6 cap 60K", salary_cap=60000)

In [ ]:
show_all()

### YOUR ANSWER - Step 4

**4a (10 points).** Fill in the points scored at each cap:
&#36;40K \_\_\_\_\_\_, &#36;45K \_\_\_\_\_\_, &#36;50K \_\_\_\_\_\_, &#36;55K \_\_\_\_\_\_, &#36;60K \_\_\_\_\_\_.

**4b (5 points).** Each extra &#36;5,000 of cap bought \_\_\_\_\_\_, then \_\_\_\_\_\_, then \_\_\_\_\_\_, then \_\_\_\_\_\_ more points.
What is the pattern, and where have you seen it before in this course? One sentence.

*your sentence*

**4c (5 points).** At the &#36;60,000 cap, the team leaves &#36;\_\_\_\_\_\_ unspent. Why does more money stop helping? One
or two sentences.

*your sentences*

# Step 5 · Add business rules (30 points)

These are the rules from the project selection videos in M5.1, written for players. Each scenario keeps the rules
before it and adds one more, so you can watch the objective fall and price each rule by the drop it causes.

| Rule | In words | Type |
|---|---|---|
| `max_per_team=2` | No more than 2 players from any one NFL team | At most n |
| `must_start=["Cooper Kupp"]` | Our owner insists on Cooper Kupp, the top-scoring receiver so far this season | Must have |
| `not_together=[("Deebo Samuel", "Stefon Diggs")]` | Samuel or Diggs, but not both | Mutually exclusive |
| `requires=[("Darrel Williams", "Tyreek Hill")]` | We can only start Williams if we also start his teammate Hill | Contingency |

Just run the four cells.

In [ ]:
pick_team("7 at most 2 per team", max_per_team=2)

In [ ]:
pick_team("8 + must start Kupp", max_per_team=2, must_start=["Cooper Kupp"])

In [ ]:
pick_team("9 + Samuel or Diggs", max_per_team=2, must_start=["Cooper Kupp"],
          not_together=[("Deebo Samuel", "Stefon Diggs")])

In [ ]:
pick_team("10 + Williams needs Hill", max_per_team=2, must_start=["Cooper Kupp"],
          not_together=[("Deebo Samuel", "Stefon Diggs")],
          requires=[("Darrel Williams", "Tyreek Hill")])

In [ ]:
show_all()

### Your turn: start your own player

Pick any player who is **not** on the base team. Use `show_players("WR")`, or `"RB"`, `"TE"` or `"FB"`, to see the
names, then type one between the quotes below and run it. Only this one rule is on, so compare it to the base
model.

In [ ]:
show_players("WR")

In [ ]:
my_player = "type a name here"

pick_team("11 my player", must_start=[my_player])

### YOUR ANSWER - Step 5

**5a (10 points).** Fill in the objective as the rules stack:
base \_\_\_\_\_\_ → at most 2 per team \_\_\_\_\_\_ → must start Kupp \_\_\_\_\_\_ → Samuel or Diggs \_\_\_\_\_\_ → Williams needs
Hill \_\_\_\_\_\_.

**5b (10 points).** Write these two rules in math, using $y$:

Samuel or Diggs, but not both: \_\_\_\_\_\_\_\_\_\_\_\_

We can only start Williams if we also start Hill: \_\_\_\_\_\_\_\_\_\_\_\_

**5c (5 points).** Which rule cost the most points, and which cost the least? Why was "at most 2 per team" so
cheap? Two sentences.

*your sentences*

**5d (5 points).** I started \_\_\_\_\_\_\_\_\_\_\_\_. Forcing him onto the team cost \_\_\_\_\_\_ points compared with the base
model. Who left the team to make room? One sentence.

*your sentence*

# Step 6 · Pick the team before the games are played (15 points)

There is a catch in everything you have done so far. We used each player's **actual Week 10 points**, and nobody
knows those when the contest opens. A real entry has to be locked in before kickoff, using a forecast.

Here is a simple forecast: each player's average points over Weeks 2 through 9. The first cell builds it. The
second cell maximizes the **forecast** instead of the actual points, then shows you what that team really scored
in Week 10.

In [ ]:
# each player's average points in the weeks before our contest
past = all_weeks[all_weeks['Week'] < 10]
past_average = past.groupby('Player')['Points'].mean()

players['Forecast'] = players['Player'].map(past_average)
players['Forecast'] = players['Forecast'].fillna(0).round(2)

players[['Player', 'Position', 'Salary', 'Forecast', 'Points']].sort_values('Forecast', ascending=False).head(10)

In [ ]:
pick_team("12 forecast from Weeks 2-9", forecast=True)

In [ ]:
show_all()

### YOUR ANSWER - Step 6

**6a (5 points).** The model expected \_\_\_\_\_\_ points. The team actually scored \_\_\_\_\_\_ points, compared with
\_\_\_\_\_\_ points for the base model in Step 3.

**6b (10 points).** Which player on the forecast team missed his forecast by the most? Why did the team score so
much less than the model expected? And what does that tell you about the 199.2 points from Step 3? Two or three
sentences.

*your sentences*

# Step 7 · Your recommendation (15 points)

A friend is entering next week's contest and has never heard of Pyomo. In **four or five sentences**, tell them:

1. what the model does well, in plain words,
2. the one number from this assignment that surprised you most, and
3. what you would improve first before putting real money on it.

### YOUR ANSWER - Step 7

*your four or five sentences*

# Turn it in

1. Click **Runtime → Run all** and wait for it to finish. Scroll down and make sure every step printed something
   and there are no red error messages.
2. Check that every **YOUR ANSWER** cell is filled in.
3. **File → Download → Download .ipynb**, and upload that file to Assignment 6 on HuskyCT.

**Due Friday, November 20 at 11:59 PM.**